# 001 Multi-agent Overview

这是 LangChain Multi-agent 学习线的第一份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/multi-agent

学习目标：

1. 理解 multi-agent 解决的不是“并发聊天”，而是上下文、职责和控制流问题
2. 认识 Subagents、Handoffs、Skills、Router、Custom workflow 五种模式
3. 判断什么时候不需要多智能体
4. 用控制权、上下文隔离、成本和验证来选择模式
5. 对比本仓库 Harness 多智能体设计中的 coordinator / subagent / verification

这一讲不调用真实模型，先建立工程判断。

## 1. 为什么不是直接多开几个 agent

多智能体系统真正要解决的问题通常是：

- 一个 agent 的上下文太长
- 工具太多，模型选择困难
- 任务职责混在一起，难以验证
- 需要不同 agent 使用不同 prompt / tools / memory
- 需要把 research、implementation、verification 分开

所以多智能体的价值不是：

```text
让更多模型同时说话
```

而是：

```text
分工、隔离、验证、综合
```

这和我们前面学 Harness 时的判断一致。

## 2. 五种模式的第一印象

| 模式 | 适合场景 | 控制权在哪里 | 上下文特点 |
| --- | --- | --- | --- |
| Subagents | 主 agent 委派局部任务 | supervisor / coordinator | 子 agent 隔离上下文，主 agent 综合 |
| Handoffs | 当前 agent 把控制权交给另一个 agent | active agent 转移 | 新 agent 接管对话 |
| Skills | 复用稳定能力包 | 主 agent 仍控制 | skill 提供流程/知识/脚本 |
| Router | 先分类，再进入专门 agent | router 控制入口 | 每个 agent 只看自己需要的上下文 |
| Custom workflow | 明确步骤和节点 | workflow 控制 | 状态和流转显式定义 |

一句话区分：

```text
Subagent 是委派；Handoff 是交接；Skill 是能力模块；Router 是入口分流；Workflow 是显式编排。
```

In [2]:
from dataclasses import dataclass
from enum import Enum


class MultiAgentPattern(str, Enum):
    SINGLE_AGENT = "single_agent"
    SUBAGENTS = "subagents"
    HANDOFFS = "handoffs"
    SKILLS = "skills"
    ROUTER = "router"
    CUSTOM_WORKFLOW = "custom_workflow"


@dataclass
class TaskProfile:
    name: str
    domains: int
    needs_user_continuation: bool
    needs_strict_order: bool
    reusable_capability: bool
    high_verification_risk: bool
    context_size: str  # small / medium / large


## 3. 什么时候不需要多智能体

单 agent 已经足够的情况：

- 任务领域单一
- 工具数量少
- 上下文不长
- 不需要独立验证
- 不需要复杂权限边界

工程上不要为了“看起来高级”拆 agent。拆分会带来额外成本：

- 更多模型调用
- 更复杂的上下文传递
- 更难的错误定位
- 更高的延迟
- 更复杂的综合逻辑

In [3]:
def choose_pattern(task: TaskProfile) -> MultiAgentPattern:
    if task.needs_strict_order and task.high_verification_risk:
        return MultiAgentPattern.CUSTOM_WORKFLOW

    if task.needs_user_continuation:
        return MultiAgentPattern.HANDOFFS

    if task.reusable_capability:
        return MultiAgentPattern.SKILLS

    if task.domains >= 3:
        return MultiAgentPattern.ROUTER

    if task.context_size == "large" or task.high_verification_risk:
        return MultiAgentPattern.SUBAGENTS

    return MultiAgentPattern.SINGLE_AGENT


examples = [
    TaskProfile("普通聊天问答", 1, False, False, False, False, "small"),
    TaskProfile("查询天气并格式化回答", 1, False, False, True, False, "small"),
    TaskProfile("用户先问售前，后续转人工客服 agent", 2, True, False, False, False, "medium"),
    TaskProfile("财务/合同/项目多领域问答入口", 4, False, False, False, False, "medium"),
    TaskProfile("代码改造并独立验证", 2, False, True, False, True, "large"),
]

for item in examples:
    print(item.name, "->", choose_pattern(item).value)


普通聊天问答 -> single_agent
查询天气并格式化回答 -> skills
用户先问售前，后续转人工客服 agent -> handoffs
财务/合同/项目多领域问答入口 -> router
代码改造并独立验证 -> custom_workflow


## 4. 控制权：谁决定下一步

多智能体设计最重要的问题之一是：谁决定下一步？

| 模式 | 谁决定下一步 |
| --- | --- |
| Subagents | coordinator / supervisor |
| Handoffs | 当前 active agent 可以交接给另一个 agent |
| Skills | 主 agent 决定是否使用 skill |
| Router | router 先决定入口 agent |
| Custom workflow | workflow 图决定下一步 |

如果你希望流程稳定、可审计、可恢复，通常更偏向 router 或 custom workflow。

如果你希望 agent 更自由地处理开放式对话，handoff 会更自然。

## 5. 上下文隔离：为什么 subagent 有价值

假设主 agent 有 20 条上下文，research 需要读 80 条代码片段。

如果全部塞给主 agent：

```text
main context = 20 + 80 = 100
```

如果交给 research subagent：

```text
research context = 80
main context = 20 + synthesis summary
```

重点不是省一点 token，而是让主 agent 不被细节淹没。

In [ ]:
def context_cost(main_messages: int, research_docs: int, summary_messages: int = 5) -> dict:
    single_agent_context = main_messages + research_docs
    multi_agent_main_context = main_messages + summary_messages
    research_agent_context = research_docs
    return {
        "single_agent_context": single_agent_context,
        "research_agent_context": research_agent_context,
        "main_after_synthesis": multi_agent_main_context,
    }


context_cost(main_messages=20, research_docs=80)


## 6. Synthesis：coordinator 不能只是转发器

多智能体系统里，worker 可以带回局部观察。

但 coordinator 必须负责综合：

```text
worker result
  -> 去重
  -> 判断可信度
  -> 形成下一步计划
  -> 给用户清晰回答
```

如果 coordinator 只是把 subagent 的输出贴回来，系统不会变可靠，只是把混乱分散到了多个地方。

In [4]:
worker_reports = [
    {"agent": "research", "finding": "approval 恢复入口是 stream_resume_approval", "confidence": "high"},
    {"agent": "verification", "finding": "测试覆盖 approve/reject/edit 三条路径", "confidence": "medium"},
]


def synthesize_reports(reports: list[dict]) -> str:
    high_confidence = [item for item in reports if item["confidence"] == "high"]
    needs_check = [item for item in reports if item["confidence"] != "high"]
    return (
        "高置信观察：" + "; ".join(item["finding"] for item in high_confidence)
        + "\n需要继续验证：" + "; ".join(item["finding"] for item in needs_check)
    )


print(synthesize_reports(worker_reports))


高置信观察：approval 恢复入口是 stream_resume_approval
需要继续验证：测试覆盖 approve/reject/edit 三条路径


## 7. 和本仓库 Harness 的对应关系

| Multi-agent 概念 | 本仓库 Harness 对应点 |
| --- | --- |
| coordinator / supervisor | 主 `HarnessChatAgent` 的 planner 和 synthesis |
| research subagent | 只读调查，限制 tools / paths |
| verification subagent | 独立检查结果是否可信 |
| implementation subagent | 受控写入，必须 approval |
| skills | `.agents/skills` 下的可复用能力包 |
| router | planner action / role 选择 |
| custom workflow | 固定 research -> implement -> verify -> synthesize 流程 |

所以多智能体不是一个全新的概念，而是把我们已经学过的 Harness 边界显式化。

## 8. 选择模式的最小问题清单

设计多智能体前，先问这几个问题：

1. 单 agent 为什么不够？
2. 是否需要隔离上下文？
3. 是否需要独立验证？
4. 是否只是复用一个稳定能力？如果是，skill 可能比 subagent 更合适。
5. 是否只是按意图分类？如果是，router 可能足够。
6. 是否需要固定顺序和审计？如果是，custom workflow 更合适。
7. 谁负责最终 synthesis？

如果第 1 个问题答不上来，不要急着做多智能体。

## 9. 本讲练习

请判断下面场景更适合哪种模式：

1. 用户问天气，系统已有稳定 weather skill。
2. 用户问“帮我分析这个 bug，并给出修复方案”，需要只读调查和独立验证。
3. 用户先和销售 agent 对话，确认购买意向后转给合同 agent。
4. 用户进入企业知识库问答系统，问题可能属于财务、合同、人事、项目四类。
5. 生产发布前必须按固定顺序执行检查、构建、测试、审批。

参考答案：

1. Skills
2. Subagents
3. Handoffs
4. Router
5. Custom workflow

## 10. 本讲小结

这一讲的核心判断：

```text
多智能体不是为了更多模型，而是为了更清晰的分工、更稳定的上下文边界和更可靠的验证综合。
```

下一讲进入 `SubagentsHandoffs`，重点区分“委派”和“交接”。